# Astra v4 — 1.2B Parameter Cyber Security Detection Training (T4 Optimized with FP16)

Trains a **1.2 Billion parameter** transformer (`astra1b_cyber`) on the NSL-KDD cyber intrusion dataset using **FP16 mixed precision** to fit within the **16GB VRAM** of a free T4 GPU.

Runtime: **Settings → Runtime → GPU (T4)**, Internet ON. Run cells top to bottom.

In [ ]:
import torch
print('torch', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0), torch.cuda.get_device_capability(0))
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'VRAM: {vram:.2f} GB')

In [ ]:
import os, subprocess
REPO = '/content/astra'
if os.path.isdir(REPO):
    subprocess.run(f'rm -rf {REPO}', shell=True, check=True)
subprocess.run(f'git clone --depth 1 https://github.com/anoneurx/astra.git {REPO}', shell=True, check=True)
assert os.path.isdir(REPO), 'clone failed - check Internet is ON and retry'
os.chdir(REPO)
print('repo ready at', os.getcwd())

**Mount Google Drive** — persistent backup:

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs('/content/drive/MyDrive/astra_checkpoints', exist_ok=True)
print('Drive ready')

---
## STAGE 0 — Build cyber BPE tokenizer

In [ ]:
import os, subprocess
os.chdir('/content/astra')
if not os.path.exists('tokenizer/artifacts/cyber_bpe.json'):
    subprocess.run('python tokenizer/train_tokenizer.py --config configs/tokenizer_cyber.json', shell=True, check=True)
print('Tokenizer artifact verified:', os.path.exists('tokenizer/artifacts/cyber_bpe.json'))

---
## STAGE 1 — Train 1.2B Cyber Model (`astra1b_cyber`) on T4 with FP16 Mixed Precision
Auto-downloads as **`astra_1b_cyber_final.npz`** and backs up to Google Drive.

In [ ]:
import os, subprocess, shutil, json
os.chdir('/content/astra')
if not os.path.exists('tokenizer/artifacts/cyber_bpe.json'):
    subprocess.run('python tokenizer/train_tokenizer.py --config configs/tokenizer_cyber.json', shell=True, check=True)

os.makedirs('configs', exist_ok=True)
config_data = {
  "model": {
    "vocab_size": 8192,
    "d_model": 2048,
    "n_layers": 24,
    "n_heads": 32,
    "d_head": 64,
    "d_ffn": 8192,
    "max_seq_len": 256,
    "rope_theta": 10000.0,
    "eps": 1e-6,
    "tie_embeddings": True,
    "pos_type": "rope",
    "ffn_type": "swiglu"
  },
  "training": {
    "max_steps": 10000,
    "peak_lr": 6e-5,
    "min_lr": 1e-6,
    "warmup_steps": 500,
    "batch_seq": 2,
    "accum_steps": 16,
    "optimizer": "adamw",
    "scheduler": "cosine",
    "weight_decay": 0.1,
    "grad_clip": 1.0,
    "val_every": 250,
    "save_every": 500,
    "val_shards": 1,
    "experiment_store": "experiments/runs"
  },
  "tokenizer": "tokenizer/artifacts/cyber_bpe.json",
  "data": {
    "train": "datasets/cyber/train.txt",
    "val": "datasets/cyber/val.txt"
  },
  "seed": 42,
  "out_dir": "checkpoints/astra1b_cyber",
  "safety": {
    "leak_check": False,
    "note": "1.2B param model on cyber corpus optimized with FP16 for T4 16GB VRAM."
  }
}
with open('configs/astra1b_cyber.json', 'w') as f:
    json.dump(config_data, f, indent=2)

os.makedirs('/content/runs', exist_ok=True)
res = subprocess.run(
    'python training/gpu_train.py --config configs/astra1b_cyber.json --out /content/runs/astra1b_cyber --cache-dir /content/cache --dtype fp16',
    shell=True, capture_output=True, text=True
)
print("STDOUT:\n", res.stdout[-3000:])
print("STDERR:\n", res.stderr[-3000:])
if res.returncode != 0:
    raise RuntimeError("Training failed — see STDERR above")

SRC = '/content/runs/astra1b_cyber/final.npz'
assert os.path.exists(SRC), '1B MODEL TRAINING FAILED'

shutil.copy(SRC, '/content/astra_1b_cyber_final.npz')
shutil.copy(SRC, '/content/drive/MyDrive/astra_checkpoints/astra_1b_cyber_final.npz')
print('1B model ok:', os.path.getsize('/content/astra_1b_cyber_final.npz'), 'bytes')
from google.colab import files
files.download('/content/astra_1b_cyber_final.npz')